In [4]:
from collections import Counter
import pandas as pd
import networkx as nx
import matplotlib as plt

In [5]:
RN = pd.read_csv("GenieLinkList_core_03_25_24_ds6.csv", index_col=0)
pred_TFs = pd.read_csv("core_network_TFs_03_25_24.csv")
n = 1111 # Get the top n edges

RN_f1_micro_max = RN.head(n).drop('weight',axis=1)

GN_micro = nx.DiGraph()
GN_micro.add_edges_from([tuple(x) for x in RN_f1_micro_max.values])
## Annotate TFs
for node in GN_micro.nodes():
    if node in pred_TFs.TFs.values:
        GN_micro.nodes[node]['TF'] = 1
    else:
        GN_micro.nodes[node]['TF'] = 0


# Step 1: Filter nodes by the 'TF' attribute
tf_nodes = [node for node, attr in GN_micro.nodes(data=True) if attr.get('TF')]

# Step 2 and 3: Count the number of edges or outgoing edges for these nodes
more_than_one_edge = 0
more_than_one_outgoing_edge = 0

for node in tf_nodes:
    # Count the total number of edges (both incoming and outgoing)
    if GN_micro.degree(node) > 1:
        print(node)
        more_than_one_edge += 1
print('\n')
for node in tf_nodes:
    # Count the number of outgoing edges for directed graphs
    if GN_micro.is_directed():
        if GN_micro.out_degree(node) > 1:
            print(node)
            more_than_one_outgoing_edge += 1

# Output the results
print(f"Number of 'TF' nodes with more than one edge: {more_than_one_edge}")
if GN_micro.is_directed():
    print(f"Number of 'TF' nodes with more than one outgoing edge: {more_than_one_outgoing_edge}")



SYNPCC7942_RS06230
SYNPCC7942_RS05680
SYNPCC7942_RS08885
SYNPCC7942_RS03075
SYNPCC7942_RS03260
SYNPCC7942_RS09380
SYNPCC7942_RS11465
SYNPCC7942_RS07435
SYNPCC7942_RS10700
SYNPCC7942_RS03325
SYNPCC7942_RS09435
SYNPCC7942_RS07185
SYNPCC7942_RS05640
SYNPCC7942_RS03160
SYNPCC7942_RS00480
SYNPCC7942_RS10350
SYNPCC7942_RS08645
SYNPCC7942_RS07805
SYNPCC7942_RS03565
SYNPCC7942_RS08820
SYNPCC7942_RS05870
SYNPCC7942_RS04225
SYNPCC7942_RS09500
SYNPCC7942_RS12560
SYNPCC7942_RS10260
SYNPCC7942_RS00645
SYNPCC7942_RS06700
SYNPCC7942_RS06745
SYNPCC7942_RS05195
SYNPCC7942_RS05070
SYNPCC7942_RS09760
SYNPCC7942_RS10040
SYNPCC7942_RS06365
SYNPCC7942_RS05230
SYNPCC7942_RS03530
SYNPCC7942_RS00455
SYNPCC7942_RS13165
SYNPCC7942_RS03440


SYNPCC7942_RS05680
SYNPCC7942_RS08885
SYNPCC7942_RS03075
SYNPCC7942_RS03260
SYNPCC7942_RS09380
SYNPCC7942_RS11465
SYNPCC7942_RS07435
SYNPCC7942_RS10700
SYNPCC7942_RS03325
SYNPCC7942_RS09435
SYNPCC7942_RS07185
SYNPCC7942_RS05640
SYNPCC7942_RS03160
SYNPCC7942_RS00480
SYNPCC7942

In [6]:
# Get components of the network
components = nx.weakly_connected_components(GN_micro)
# Select the larget component
largest_component = max(components, key=len)
#Construct a network from the largest component
H = GN_micro.subgraph(largest_component)


#Get the number of edges for each node
node_degree = dict(H.degree())
#Determine betweeness centrality for all nodes in the graph
bc = nx.betweenness_centrality(H)
#Determine betweeness centrality for all nodes in the graph
cc = nx.closeness_centrality(H)
##Determine the percolation centrality of nodes
ec = nx.eigenvector_centrality(H, max_iter=1000)

## Append these node properties to the graph
nx.set_node_attributes(H, node_degree, "degree")
nx.set_node_attributes(H, bc, "betweeness centrality")
nx.set_node_attributes(H, cc, "closeness centrality")
nx.set_node_attributes(H, ec, "eigenvector centrality")

core_dict = nx.core_number(H)
core_count_dict = {}
#core_number returns the k core for a give node, to determine node count at each cutoff level:
#Start with the highest degree core and append that value to a dictionary
#For each subsequent core add the number of values in that core to the core counted before 
for i in sorted(list(set(core_dict.values())), reverse=True):
    if core_count_dict == {}:
        core_count_dict[i] = Counter(core_dict.values())[i]
    else:
        core_count_dict[i] = Counter(core_dict.values())[i] + core_count_dict[i+1]
        
nx.set_node_attributes(H, core_dict, "k-core")
for key, value in core_count_dict.items():
    print(f"{value} genes assigned to k-core value of {key}")

def community_detection_node_map(G, clusters, min_size):

    cluster_dict = {}
    for i, cluster in enumerate(clusters):
        if len(cluster) > min_size:
            for gene in cluster:
                cluster_dict[gene] = i+1
        else:
            for gene in cluster:
                cluster_dict[gene] = 0
            
    return(cluster_dict)

louv = nx.community.louvain_communities(H, resolution=5, seed=12)
louv_map = community_detection_node_map(H, louv, min_size=9)

nx.set_node_attributes(H, louv_map, "louvian_cluster")
selected_nodes = [n for n,v in H.nodes(data=True) if v["louvian_cluster"] == 4]  
#selected_edges = [(u,v) for u,v,e in H.edges(data=True) if e['louvian_cluster'] == 4]

nx.write_graphml(selected_nodes, "Abrb4.graphml")

18 genes assigned to k-core value of 3
223 genes assigned to k-core value of 2
889 genes assigned to k-core value of 1


AttributeError: 'list' object has no attribute 'is_directed'